## Chosing The right Regression model..

In [2]:
import pandas as pd 
house_price=pd.read_csv("z.Data/house_price_bd.csv")
house_price.sample(n=7)

,Title,Bedrooms,Bathrooms,Floor_no,Occupancy_status,Floor_area,City,Price_in_taka,Location
3657,Visit This Large 5 Katha Plot Is For Sale In B...,NaN,NaN,NaN,vacant,3600.0,narayanganj-city,"৳5,000,000","Rupganj, Narayanganj"
2601,Grab This 240 Sq Ft Shop For Sale At Bakalia,NaN,NaN,3,vacant,240.0,chattogram,"৳5,520,000","Dewan Bazar, Bakalia"
2862,This Beautiful 1800 Sq Ft Flat Located In Nasi...,3.0,4.0,10,vacant,1800.0,chattogram,"৳12,500,000","Nasirabad, Bayazid"
3583,A Residential Plot For Sale In The Location Of...,NaN,NaN,NaN,vacant,3600.0,narayanganj-city,"৳6,000,000","Rupganj, Narayanganj"
2298,Good-looking Flat Is Vacant For Sale In Dewan ...,3.0,3.0,1,vacant,1253.0,chattogram,"৳5,638,500","Dewan Bazar, Bakalia"
796,"At North Pirerbag , A 1250 Sq Ft Well Fitted R...",3.0,3.0,9,vacant,1250.0,dhaka,"৳6,500,000","Pirerbag, Mirpur"
3273,3 Katha nice Plot is available for sale in Bpr...,NaN,NaN,NaN,vacant,2160.0,narayanganj-city,"৳4,575,000","Rupganj, Narayanganj"


  ### Let replace the ৳(string) to Numarical value 

In [3]:
house_price["Price_in_taka"]=house_price["Price_in_taka"].str.replace("৳", "").str.replace(",", "").astype(float)


### Now make other numaric 

In [4]:
x= house_price.drop( "Price_in_taka", axis=1)
y= house_price["Price_in_taka"]

### OK see what model we can chose 

In [5]:
house_price.isna().sum()

Title                  0
Bedrooms            1001
Bathrooms           1001
Floor_no             684
Occupancy_status      99
Floor_area            99
City                   0
Price_in_taka          0
Location               6
dtype: int64

### Hm we see . So many missing. Okay then fillup 

In [6]:
import numpy as np
import pandas as pd

# First, convert these columns to numeric (removes non-numeric values and converts to NaN)
house_price["Bedrooms"] = pd.to_numeric(house_price["Bedrooms"], errors='coerce')
house_price["Bathrooms"] = pd.to_numeric(house_price["Bathrooms"], errors='coerce')
house_price["Floor_no"] = pd.to_numeric(house_price["Floor_no"], errors='coerce')

# Now fill NaNs with random integers
house_price.loc[house_price["Bedrooms"].isna(), "Bedrooms"] = np.random.randint(
    1, int(house_price["Bedrooms"].max()) + 1, 
    size=house_price["Bedrooms"].isna().sum()
)  

house_price.loc[house_price["Bathrooms"].isna(), "Bathrooms"] = np.random.randint(
    1, int(house_price["Bathrooms"].max()) + 1, 
    size=house_price["Bathrooms"].isna().sum()
)

house_price.loc[house_price["Floor_no"].isna(), "Floor_no"] = np.random.randint(
    1, int(house_price["Floor_no"].max()) + 1, 
    size=house_price["Floor_no"].isna().sum()
)

# Fill categorical and continuous columns
house_price["Occupancy_status"]=house_price["Occupancy_status"].fillna("Occupied")
house_price["Floor_area"]= house_price["Floor_area"].fillna(house_price["Floor_area"].mean())  # Fixed typo

# Fill Location with random choice
area = ["Gandaria, Sutrapur", "Middle Badda, Badda", "Madhya Ajampur, Dakshin Khan", "Section 6, Mirpur"]
house_price.loc[house_price["Location"].isna(), "Location"] = np.random.choice(
    area, 
    size=house_price["Location"].isna().sum(),
    p=[0.3, 0.2, 0.1, 0.4]
)

# Check for remaining NaNs
print(house_price.isnull().sum())

# '>=' not supported between instances of 'str' and 'float' 

Title               0
Bedrooms            0
Bathrooms           0
Floor_no            0
Occupancy_status    0
Floor_area          0
City                0
Price_in_taka       0
Location            0
dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Make sure house_price is fully cleaned (run your cleaning code first)
# Then create x and y from the cleaned data
x = house_price.drop('Price_in_taka', axis=1)  # Features (assuming Price_in_taka is target)
y = house_price['Price_in_taka']                # Target

# Convert categorical columns to numeric using get_dummies
x = pd.get_dummies(x, drop_first=True)

# IMPORTANT: Handle any remaining NaNs
x = x.fillna(x.mean())  # Fill remaining NaNs with column mean

# Check for any remaining NaNs
print("NaNs remaining in x:")
print(x.isnull().sum())

# Now split the data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Train the model
from sklearn.linear_model import Ridge
model = Ridge() 
model.fit(x_train, y_train) 
print(f"R² Score: {model.score(x_test, y_test)}")


NaNs remaining in x:
Bedrooms                                                  0
Bathrooms                                                 0
Floor_no                                                  0
Floor_area                                                0
Title_&#039;Full building sale in Basundhara&#039;        0
                                                         ..
Location_Yakub Nagar Road, 33 No. Firingee Bazaar Ward    0
Location_Zakir Hosain Road, Mohammadpur                   0
Location_Zakir Hossain By Lane, East Nasirabad            0
Location_Zakir Hossain Road, Khulshi                      0
Location_aziz Moholla, Mohammadpur                        0
Length: 3433, dtype: int64
R² Score: 0.7464095595160873


In [11]:
from sklearn.impute import SimpleImputer

x = pd.get_dummies(x, drop_first=True) 

# Use SimpleImputer for numeric columns
imputer = SimpleImputer(strategy='mean')
x_imputed = imputer.fit_transform(x)
x = pd.DataFrame(x_imputed, columns=x.columns)

# Now proceed with train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

from sklearn.ensemble import RandomForestRegressor 
mdl =RandomForestRegressor()
mdl.fit(x_train, y_train)
mdl.score(x_test, y_test)

0.8888695529095153